In [ ]:
import pyautogui as pyg
from PIL import ImageGrab
import time

pyg.FAILSAFE = True

print("=== 恐龍全自動 AI（臨測起跳 / 修正過早落地版）===")
print("請確保 Chrome 視窗最大化，按 Ctrl+0 將縮放定在 100%")
print("3 秒後自動啟動...\n")

for i in range(3, 0, -1):
    print(f"{i}...")
    time.sleep(1)

# 1. 計算螢幕與圖像倍率
pyg_w, pyg_h = pyg.size()
init_img = ImageGrab.grab()
img_w, img_h = init_img.size
scale = img_w / pyg_w

# 2. 強制聚焦 Chrome
pyg.click(pyg_w // 2, pyg_h // 2)
pyg.press('space')
time.sleep(0.3)

# 3. 搜尋小恐龍本體
img = ImageGrab.grab()
dino_x, dino_y = None, None

for y in range(int(img_h * 0.2), int(img_h * 0.8), 3):
    for x in range(int(img_w * 0.05), int(img_w * 0.4), 3):
        p = img.getpixel((x, y))
        if p[0] < 120 and p[1] < 120 and p[2] < 120:
            dino_x, dino_y = x, y
            break
    if dino_x:
        break

if not dino_x:
    print("❌ 找不到小恐龍！請確認 Chrome 畫面沒有被遮擋。")
else:
    # 🎯 關鍵修正：將 OFFSET_X 從 60 大幅縮短至 42，貼近恐龍，解決「太早跳」
    OFFSET_X = 42  
    left = int(dino_x + (OFFSET_X * scale))
    right = int(dino_x + ((OFFSET_X + 10) * scale))
    
    # 懸空防線：離地 Dino_Y - 8，絕不踩地面碎石
    scan_box = (left, int(dino_y - (32 * scale)), right, int(dino_y - (8 * scale)))

    print(f"✅ 定位成功！Dino:({dino_x}, {dino_y}) | 偵測距離: {OFFSET_X}px")
    print("--- 極近距離精準起跳監控中... ---")

    try:
        while True:
            crop = ImageGrab.grab(bbox=scan_box)
            w, h = crop.size
            bg = crop.getpixel((0, 0))[0]
            is_night = bg < 120

            min_x, max_x = w, 0
            min_y, max_y = h, 0
            dark_pixels = 0

            # 遍歷區域
            for bx in range(0, w, 2):
                for by in range(0, h, 2):
                    val = crop.getpixel((bx, by))[0]
                    if (is_night and val - bg > 25) or (not is_night and bg - val > 25):
                        dark_pixels += 1
                        if bx < min_x: min_x = bx
                        if bx > max_x: max_x = bx
                        if by < min_y: min_y = by
                        if by > max_y: max_y = by

            # 🎯 發現障礙物
            if dark_pixels >= 4:
                obj_width = max_x - min_x + 1
                obj_height = max_y - min_y + 1

                # 1. 障礙物向下延伸超過感應框 60% -> 【跳躍】
                if max_y > int(h * 0.60):
                    pyg.press('space')
                    print(f"【🦘 精準跳躍 (實體進場)】Pts:{dark_pixels} MaxY:{max_y}/{h}   ", end="\r")
                    time.sleep(0.08)  # 縮短冷卻時間

                # 2. 懸空且橫向展翅 -> 【蹲下】
                elif obj_width >= obj_height:
                    pyg.keyDown('down')
                    time.sleep(0.18)
                    pyg.keyUp('down')
                    print(f"【🦅 蹲下 (懸空翼龍)】Pts:{dark_pixels} MaxY:{max_y}/{h}   ", end="\r")
                    time.sleep(0.04)

                # 3. 懸空但呈現縱向型特徵（高仙人掌頂端） -> 【預備跳躍】
                else:
                    pyg.press('space')
                    print(f"【🦘 跳躍 (高仙人掌頂端)】Pts:{dark_pixels} MaxY:{max_y}/{h}   ", end="\r")
                    time.sleep(0.08)

    except KeyboardInterrupt:
        print("\n自動化程式已停止。")

=== 恐龍全自動 AI（臨測起跳 / 修正過早落地版）===
請確保 Chrome 視窗最大化，按 Ctrl+0 將縮放定在 100%
3 秒後自動啟動...

3...
2...
1...
✅ 定位成功！Dino:(141, 339) | 偵測距離: 42px
--- 極近距離精準起跳監控中... ---
